**Машинное обучение в экономике**

**Консультация. Генерация данных**

Установка библиотек

In [ ]:
# !python.exe -m pip install --upgrade pip
# !pip install numpy
# !pip install pandas
# !pip install scikit-learn
# !pip install openpyxl
# !pip install doubleml

In [ ]:
# Подключим необходимые библиотеки
import numpy as np
import pandas as pd
import scipy as scipy
from copy import deepcopy
import math
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
from scipy.stats import t, chi2, f, norm, poisson, binom, uniform, expon, logistic
import seaborn
import networkx as nx

**Описание сюжета** 😸

**Целевая переменная** - зарплата $\text{Wage}_{i}$.

**Переменная воздействия** - факт наличия высшего образования $\text{Educ}_{i}$ (1 - есть, 0 - нет).

**Инструментальная переменная** - факт наличия высшего образования хотя бы у одного из родителей $\text{Parents}_{i}$ (1 - есть, 0 - нет).

**Контрольные переменные**:

*   $\text{Experience}_{i}$ - стаж работы в годах.
*   $\text{City}_{i}$ - факт проживания в городе (1 - живет в городе, 0 - живет не в городе).
*   $\text{Married}_{i}$ - семейный статус (1 - в браке, 0 - холостой).

**Ненаблюдаемые переменные, порождающие эндогенность**:

*  $\text{Abilities}_{i}$ - способности индивида, не обусловленные получением образования, например, врожденные или приобретенные в школе.







In [ ]:
# Визуализируем предполагаемые связи

# Стиль отображения графика
nx_options = {
    'node_color': 'white',
    'node_size': 2000,
    'width': 3,
    'arrowstyle': '-|>',
    'arrowsize': 12,
}

# Связи
edges = [
    ('Educ', 'Wage'),         # (откуда стрелочка, куда стрелочка)
    ('Experience', 'Wage'),
    ('City', 'Wage'),
    ('Married', 'Wage'),
    ('Abilities', 'Wage'),
    ('Abilities', 'Educ'),
    ('Parents', 'Educ'),

]

# Инициализируем граф
G = nx.DiGraph(directed = True)
G.add_edges_from(edges)

# Отображаем граф
np.random.seed(8)
nx.draw_networkx(G, arrows = True, **nx_options)

**Генерация контрольных переменных** 😸

Будем генерировать опыт работы $\text{Experience}_{i}$ взяв за основу нормальное распределение $\text{N}\left(25, 10^2\right)$.

Техническое примечание ⚡

Функция `scipy.stats.norm.rvs()` позволяет сгенерировать `size` независимых случайных величин из нормального распределения с математическим ожиданием `loc` и стандартным отклонением (квадратный корень из дисперсии) `scale`.

In [ ]:
# Для воспроизводимости
np.random.seed(123)

In [ ]:
# Число наблюдений
n = 100000

In [ ]:
# Предположим, что распределение опыта работы схоже с нормальным
experience = norm.rvs(size = n, loc = 25, scale = 10)

# Для красоты ограничим минимальные и максимальные значения
experience[experience >= 60] = 60
experience[experience <= 1]  = 1

# Также, для удобства введем округление
experience = np.round(experience)

# Посмотрим на несколько первых значений
print(experience[0:10])

In [ ]:
# Посмотрим на распределение
seaborn.histplot(experience,                 # данные
                 stat = 'density',           # тип гистограммы
                 color = "palevioletred",    # цвет гистограммы
                 label = "histogram",        # наименование гистограммы
                 bins = 100)                 # число столбиков в Гистограмме

Поскольку $\text{City}_{i}$ является бинарной переменной, принимающей значения $0$ и $1$, то она имеет распределение Бернулли $\text{City}_{i}\sim\text{Ber}(p)$. Предположим, что в городе живут $70\%$ индивидов, откуда $p=0.7$.

Техническое примечание ⚡

Функция `np.random.binomial()` позволяет сгенерировать `size` независимых случайных величин из биномиального распределения с параметрами `n` и `p`. Напомним, что распределение Бернулли является частным случаем биномиального распределения при `n = 1`.

In [ ]:
# Сгенерируем переменную на факт проживания в городе
city = np.random.binomial(n = 1, p = 0.7, size = n)

# Посмотрим на первые несколько значение
print(city[0:10])

In [ ]:
# Доля индивидов, проживающих в городе
print(np.mean(city))

По аналогии сгенерируем переменную на брак, предполагая $\text{Married}_{i}\sim\text{Ber}\left(0.6\right)$.

In [ ]:
# Сгенерируем переменную на брак
married = np.random.binomial(n = 1, p = 0.6, size = n)

# Доля индивидов, состоящих в браке
print(np.mean(married))

Для простоты мы сгенерировали контрольные переменные как независимые. Однако, при генерации можно также предполагать наличие связи между переменными. Для этого, например, можно воспользоваться [копулами](https://pypi.org/project/copulas/).

**Генерация ненаблюдаемой переменной, порождающей эндогенность** 😸

Сгенерируем способности $\text{Abilities}_{i}$ взяв за основу распределение Стьюдента с $8$ степенями свободы.

Эта переменная будет играть ролль ненаблюдаемой, отсутствие в данных которой и приводит к проблеме эндогенности.

Техническое примечание ⚡

Функция `scipy.stats.t.rvs()` позволяет сгенерировать `size` независимых случайных величин из распределения Стьюдента c `df` степенями свободы.

In [ ]:
# Генерируем способности
abilities = t.rvs(size = n, df = 8)

# Приведем способности к более удобной шкале
abilities = (abilities) * 10 + 50

# Также, для удобства введем округление и возьмем модуль
abilities = np.round(np.abs(abilities) + 1)

# Для удобства ограничим максимальное и минимальное значения
abilities[abilities >= 100] = 100
abilities[abilities <= 1]   = 1

# Посмотрим на несколько первых значений
print(abilities[0:10])

In [ ]:
# Посмотрим на распределение способностей
seaborn.histplot(abilities,                  # данные
                 stat  = 'density',          # тип гистограммы
                 color = "palevioletred",    # цвет гистограммы
                 label = "histogram",        # наименование гистограммы
                 bins  = 50)                 # число столбиков в Гистограмме

**Генерация инструментальной переменной** 😸

Для того, чтобы сгенерировать бинарную переменную как функцию от других переменных, необходимо сперва предположить форму условных вероятностей. Для этого удобно применять следующий алгоритм:

1.   Записать **индекс**, который отражает статистические связи инструментальной переменной с контрольными переменными.
2.   Сформировать условные вероятности взяв функцию распределения от этого индекса

Например, предположим, что условная вероятность факта наличия высшего образования у родителей положительно связана с опытом работы, проживанием в городе и браком:

$$\text{P}\left(\text{Parents}_{i} = 1|\text{Experience}_{i}, \text{City}_{i}, \text{Married}_{i}\right) = \Phi\left(\underbrace{\frac{0.3 * \text{Experience}_{i}}{ (6 - \text{City}_{i} - \text{Married}_{i})} + \text{City}_{i}\times\text{Married}_{i} - 2.5}_{\text{индекс}}\right)$$

Где $\Phi()$ - функция распределения стандартного нормального распределения.

**Важно** - для удобства генерации рекомендуется предположить, что условные вероятности инструментальной переменной должны зависеть лишь от контрольных переменных.

In [ ]:
# Сформируем индекс
parents_index = 0.3 * experience / (6 - city - married) + city * married - 2.5

In [ ]:
# Создадим условные вероятности
parents_prob = norm.cdf(parents_index)

# Посмотрим на несколько первых условных вероятностей
print(parents_prob[0:10])

In [ ]:
# Сгенерируем наличие образования у родителей
parents = np.random.binomial(n = 1, p = parents_prob, size = n)

# Посмотрим на несколько сгенерированных значений
print(parents[0:10])

Желательно, чтобы дисперсия индекса отличалась от дисперсии распределения, чья функция распределения используется на втором шаге алгоритма, не более, чем в $2$ раза. В противном случае оценки параметров такой модели могут оказаться либо слишком точными, либо слишком неточными.

In [ ]:
# Дисперсия индекса
print(np.var(parents_index))

Убедитесь, что доля единиц в сформированной переменной является адекватной. В случае наличия слишком малого или слишком большого числа единиц можно, например, отнять или прибавить константу к индексу.

In [ ]:
# Доля родителей с высшим образованием
print(np.mean(parents))

В данном случае статистические связи не обязательно отражают причинно-следственные. Поэтому в том числе возможна интерпретация с точки зрения обратной причинности. Например, можно предположить, что положительная связь между наличием у родителей высшего образования и опытом работы может объясняться тем, что люди с высшим образованием чаще обладают возможностями помочь своим детям с трудоустройством.

**Генерация переменной воздействия** 😸

Удобно предположить, что условные вероятности переменной воздействия зависят от контрольных переменных, инструментальной переменной и ненаблюдаемой переменной.

$$P(\text{Educ}_{i} = 1|\text{Experience}_{i}, \text{City}_{i}, \text{Married}_{i}, \text{Abilities}_{i}, \text{Parents}_{i}) = \\ = F_{\text{Logistic}}\left(6\times \ln\left(\text{Abilities}_{i} + 1\right) + \sqrt{\text{Experience}_{i}} + \text{City}_{i}\times\text{Married}_{i} - 30 + 3\times\text{Parents}_{i}\right)$$

Где $F_{\text{Logistic}}$ - функция распределения стандартного логистического распределения.

Для краткости введем обозначение для условной вероятности наличия образования у индивида при конкретном образовании родителей:

$$p_{k}^{\text{Parents}_{i}} = P(\text{Educ}_{i} = 1|\text{Experience}_{i}, \text{City}_{i}, \text{Married}_{i}, \text{Abilities}_{i}, \text{Parents}_{i} = k)\text{, где }k\in\{0,1\}$$

Для того, чтобы впоследствии анализировать локальные средние эффекты воздействия $\text{LATE}$, необходимо различать величину переменной воздействия $\text{Educ}_{i}$ в зависимости от значения инструмента $\text{Parents}_{i}$. Для этого рассмотрим ни от чего не зависящую равномерную случайную величину $U_{i}\sim U(0,1)$ и введем гипотетические переменные:

$$\text{Educ}_{1i} = I(p_{1}^{\text{Parents}_{i}}\geq U_{i})$$

$$\text{Educ}_{0i} = I(p_{0}^{\text{Parents}_{i}}\geq U_{i})$$


$$I(\text{условие}) = \begin{cases}1\text{, если условие выполнено}\\0\text{, в противном случае}\end{cases}$$

Переменные $\text{Educ}_{1i}$ и $\text{Educ}_{0i}$ отражают потенциальные уровни образования индивида в зависимости от наличия образования у родителей.

In [ ]:
# Равномерные случайные величины
u = uniform.rvs(size = n)

In [ ]:
# Сгенерируем часть индекса, не зависящую от образования родителей
educ_index = 6 * np.log(abilities + 1) + np.sqrt(experience) + city * married - 30

In [ ]:
# Симулируем уровень образования индивидам в случае,
# когда у их родителей есть высшее образование
parents1    = 1
educ1_index = educ_index + 3 * parents1
educ1_prob  = logistic.cdf(educ1_index)
educ1       = (educ1_prob >= u).astype(int)

# Доля людей с высшим образованием в случае, когда у всех есть
# родители с высшим образованием
np.mean(educ1)

In [ ]:
# Симулируем уровень образования индивидам в случае,
# когда у их родителей нет высшего образования
parents0    = 0
educ0_index = educ_index + 3 * parents0
educ0_prob  = logistic.cdf(educ0_index, scale = 1)
educ0       = (educ0_prob >= u).astype(int)

# Доля людей с высшим образованием в случае, когда у всех есть
# родители с высшим образованием
np.mean(educ0)

Индивидов можно разделить на $4$ группы:


*   **Always takers** - те, у кого $\text{Educ}_{0i}=\text{Educ}_{1i}=1$: получают образование независимо от образования родителей.
*   **Never takers**- те, у кого $\text{Educ}_{0i}=\text{Educ}_{1i}=0$: не получают образование независимо от образования родителей.
*   **Compliers** - те, у кого $\text{Educ}_{1i}=1$ и $\text{Educ}_{0i}=0$, то есть $\text{Educ}_{1i} > \text{Educ}_{0i}$: получают образование лишь в случае, если оно есть у родителей.
*   **Deniers** - те, у кого $\text{Educ}_{1i}=0$ и $\text{Educ}_{0i}=1$, то есть $\text{Educ}_{1i} < \text{Educ}_{0i}$: получают образование лишь в случае, если его нет у родителей.

Для соблюдения предпосылок используемых методов важно отсутствие Deniers, что гарантируется используемым процессом генерации данных.

In [ ]:
# Рассмотрим различные группы индивидов
ind_type = np.empty(n, dtype = 'U25')
ind_type[(educ1 == 1) & (educ0 == 1)] = 'Always taker'
ind_type[(educ1 == 0) & (educ0 == 0)] = 'Never taker'
ind_type[educ1 > educ0]               = 'Complier'
ind_type[educ1 < educ0]               = 'Denier'

# Посмотрим на распределение индивидов разного типа
print(pd.value_counts(ind_type))

In [ ]:
# Сравним уровни образования для одного и того же индивида в случаях,
# когда у его родителей нет высшего образования и когда оно у них есть
print(pd.DataFrame(data    = np.array([educ0, educ1, ind_type]).transpose(),
                   columns = ['educ0', 'educ1', 'Тип индивида']))

Наблюдаемый (в данных) уровень образования можно выразить как:

$$\text{Educ}_{i} = \begin{cases}\text{Educ}_{1i}\text{, если }\text{Parents}_{i} = 1\\ \text{Educ}_{0i}\text{, если }\text{Parents}_{i} = 0\end{cases} = \\ = \text{Educ}_{1i}\times\text{Parents}_{i} + \text{Educ}_{0i}\times\left(1 - \text{Parents}_{i}\right)$$

In [ ]:
# Факт наличия у индивида высшего образования
educ = educ1 * parents + educ0 * (1 - parents)

# Доли людей с высшим образованием
print(np.mean(educ))

**Важно**

*   При слабой корреляции между $\text{Educ}_{i}$ и $\text{Abilities}_{i}$ проблема эндогенности окажется несущественной, а при слишком большой скорректировать эндоенность окажется чрезвычайно сложно.
*   При слабой корреляции между $\text{Educ}_{i}$ и $\text{Parents}_{i}$ инструмент не будет релевантным и поэтому не позволит скорректировать эндогенность.

Таким образом, желательно сделать так, чтобы корреляции находились в некотором разумном диапазоне:

$$0.8\geq|\text{Corr}\left(\text{Educ}_{i}, \text{Abilities}_{i}\right)|\geq0.2$$

$$0.8\geq|\text{Corr}\left(\text{Educ}_{i}, \text{Parents}_{i}\right)|\geq0.2$$

In [ ]:
# Рассмотрим корреляции
print(pd.DataFrame(data    = [np.corrcoef(educ, abilities)[0, 1],
                              np.corrcoef(educ, parents)[0, 1]],
                   index   = ['Corr(educ, abilities)',
                              'Corr(educ, parents)'],
                   columns = ['Оценка']))

**Генерация целевой переменной** 😸

Сформируем представления о формировании потенциальной зарплаты в зависимости от наличия высшего образования.

**Основная идея**- отдача от стажа и способностей выше в случае, когда индивид получает высшее образование.

Уравнение зарплаты при отсутствии высшего образования:

$$\text{Wage}_{0i} = \underbrace{\underbrace{0.6\times\text{Abilities}_{i}}_{g_{0}^{\text{unobs}}} + \underbrace{11\times\frac{\text{Experience}_{i}}{10 - \text{City}_{i} + \text{Married}_{i}}}_{g_{0}^{\text{obs}}}}_{g_{0}} + \varepsilon_{0i}\text{, где }\varepsilon_{0i}\sim \left(8\times t(15)\right)$$

Уравнение зарплаты при наличии высшего образования:

$$\text{Wage}_{1i} = \underbrace{\underbrace{0.9\times \text{Abilities}_{i}}_{g_{1}^{\text{unobs}}} + \underbrace{12\times \frac{\text{Experience}_{i}}{9 - \text{City}_{i} + \text{Married}_{i}}}_{g_{1}^{\text{obs}}}}_{g_{1}} + \varepsilon_{1i}\text{, где }\varepsilon_{1i}\sim \left(\text{EXP}(0.1) - 10\right)$$

Наблюдаемая заработная плата:

$$\text{Wage}_{i} = \begin{cases}\text{Wage}_{1i}\text{, если }\text{Educ}_{i}=1\\ \text{Wage}_{0i}\text{, если }\text{Educ}_{i}=0\end{cases} = \\ =\text{Wage}_{1i}\times\text{Educ}_{i} + \text{Wage}_{0i}\times\left(1-\text{Educ}_{i}\right)$$

In [ ]:
# Случайные ошибки
error0 = t.rvs(size = n, df = 15) * 8
error1 = expon.rvs(size = n, scale = 10) - 10

# Функция от контрольных переменных
  # когда у индивида нет высшего образования
g0_obs   = 11 * experience / (10 - city - married)
g0_unobs =  0.6 * abilities
g0       = g0_obs + g0_unobs
  # когда у индивида есть высшее образование
g1_obs   = 20 + 12 * experience / (9 - city - married)
g1_unobs =  0.9 * abilities
g1       = g1_obs + g1_unobs

# Зарплата в зависимости от наличия высшего образования
wage0 = g0 + error0
wage1 = g1 + error1

# Наблюдаемая зарплата
wage = wage1 * educ + wage0 * (1 - educ)

Во избежание чрезвычайно точных или крайне неточных оценок, желательно, чтобы при каждом $j\in\{0, 1\}$ дисперсии $\varepsilon_{ji}$, $g_{j}$, $g_{j}^{\text{obs}}$ и  $g_{j}^{\text{unobs}}$ различались не более, чем в $5$ раз.

In [ ]:
# Приблизительно оценим адекватность дисперсий
print(pd.DataFrame(data    = [np.var(error0),   np.var(g0),
                              np.var(g0_obs),   np.var(g0_unobs),
                              np.var(error1),   np.var(g1),
                              np.var(g1_obs),   np.var(g1_unobs)],
                   index   = ['Var(eps0)',     'Var(g0)',
                              'Var(g0_obs)',   'Var(g0_unobs)',
                              'Var(eps1)',     'Var(g1)',
                              'Var(g1_obs)',   'Var(g1_unobs)'],
                   columns = ['Оценка']))

**Объединение данных** 😸

In [ ]:
# Аггрегируем данные в датафрейм
df = pd.DataFrame({'wage': wage, 'educ': educ,
                   'experience': experience, 'married': married,
                   'city': city,'parents': parents})
df = df.loc[0:n, :]

# Посмотрим на симулированные данные
df.head(10).style.format(precision = 2)

**Оценивание эффектов воздействия с помощью потенциальных исходов** 🐱

Эффект воздействия:

$$\text{TE}_{i} = \text{Wage}_{1i} - \text{Wage}_{0i}$$

In [ ]:
# Настоящие эффекты воздействия (не наблюдаются в данных)
TE = wage1 - wage0
print(TE[0:10])

Средний эффект воздействия:

$$\text{ATE} = \text{E}\left(\text{Wage}_{1i} - \text{Wage}_{0i}\right)$$

Если бы у нас были данные о $\text{Wage}_{1i}$ и $\text{Wage}_{0i}$, то мы могли бы очень точно оценить $\text{ATE}$ как:

$$\widehat{\text{ATE}} = \frac{1}{n}\sum\limits_{i=1}^{n}\text{Wage}_{1i} - \text{Wage}_{0i}$$

In [ ]:
# Точное приближение среднего эффекта воздействия, то есть
# с помощью оценки, недоступной с помощью реальных данных
ATE = np.mean(TE)
print(ATE)

Локальный средний эффект воздействия:

$$\text{LATE} = \text{E}(\text{Wage}_{1i} - \text{Wage}_{0i} | \text{Educ}_{1i} > \text{Educ}_{0i})$$

In [ ]:
# Точное приближение локального среднего эффекта воздействия, то есть
# с помощью оценки, недоступной с помощью реальных данных
LATE = np.mean(TE[ind_type == "Complier"])
print(LATE)

Условный средний эффект воздействия:

$$\text{CATE}_{i} = \text{E}\left(\text{Wage}_{1i}|X_{i}\right) - \text{E}\left(\text{Wage}_{0i}|X_{i}\right) = g_{1}(X_{i}) - g_{0}(X_{i})$$

In [ ]:
# Значения локальных средних эффектов воздействия
CATE = g1 - g0
print(CATE[0:10])

**Оценивание ATE как разницы в средних** 🐱

Наивный подход предполагает оценивание $\text{ATE}$ как средней разницы в зарплатах людей с высшим образованием и без высшего образования.

$$\widehat{\text{ATE}}_{\text{naive}} = \frac{1}{n_{1}}\sum\limits_{i:\text{Educ}_{i}=1}\text{Wage}_{1i} - \frac{1}{n_{0}}\sum\limits_{i:\text{Educ}_{i}=0}\text{Wage}_{0i}$$

In [ ]:
# Наивная оценка как разница в выборочных средних
ATE_naive = np.mean(wage[educ == 1]) - np.mean(wage[educ == 0])

# Сравнение точного приближения и наивной оценки
# когда у его родителей нет высшего образования и когда оно у них есть
print(pd.DataFrame(data    = [ATE, ATE_naive],
                   index   = ['Точная оценка с помощью потенциальных исходов',
                              'Наивная оценка с помощью наблюдаемых исходов'],
                   columns = ['ATE']))

**Цель** - используя ненаблюдаемые в данных потенциальные исходы оценить $\text{ATE}$ точнее, чем наивным методом, используя для этого синтез методов эконометрического анализа и машинного обучения.

**Дополнительный сюжет о генерации случайных величин из непрерывных распределений** 😸

Чтобы превратить случайную величину $X$ из непрерывного распределения $D_{0}$ в случайную величину $Y$ с непрерывным распределением $D_{1}$ можно воспользоваться следующей техникой:

$$Y = F^{-1}_{D_{1}}\left(F_{D_{0}}\left(X\right)\right)$$

Где:

*   $F_{D_{0}}(.)$ - функция распределения $D_{0}$.
*   $F_{D_{1}}^{-1}(.)$ - квантильная функция $D_{1}$, то есть функция, обратная функции распределения.

Если $D_{1}$ является дискретным распределением, то в качестве квантили

**Важно** - с помощью этой техники можно генерировать случайные величины из равномерного распределения и превращать их в случайные величины из любых других распределений.

**Доказательство**:

Покажем, что после обозначенных преобразований $Y\sim D_{1}$. Для этого достаточно показать, что в точке $t\in R$ функция распределения $Y$ совпадает с функцией распределения $D_{1}$, то есть $F_{Y}(t)=F_{D_{1}}(t)$.

Поскольку функция распределения и квантильная функция не убывают, то их взятие с левой и правой сторон неравенства никак его не изменяет:

$$F_{Y}(t) = P\left(Y\leq t\right) = P\left(F^{-1}_{D_{1}}\left(F_{D_{0}}\left(X\right)\right)\leq t\right) = P\left(F_{D_{1}}\left(F^{-1}_{D_{1}}\left(F_{D_{0}}\left(X\right)\right)\right)\leq F_{D_{1}}(t)\right) = P\left(F_{D_{0}}(X)\leq F_{D_{1}}(t)\right) = P\left(F_{D_{0}}^{-1}\left(F_{D_{0}}(X)\right)\leq F_{D_{0}}^{-1}\left(F_{D_{1}}(t)\right)\right) = P(X\leq F_{D_{0}}^{-1}\left(F_{D_{1}}(t)\right)) = F_{X}(F_{D_{0}}^{-1}\left(F_{D_{1}}(t)\right)) = F_{D_{0}}(F_{D_{0}}^{-1}\left(F_{D_{1}}(t)\right)) = F_{D_{1}}(t)$$

**Пример**

Сгенерируем случайные величины $X$ из распределения $D_{0} = \text{N}(20, 100)$.

Функция `scipy.stats.norm.rvs()` позволяет сгенерировать `size` независимых случайных величин из нормального распределения с математическим ожиданием `loc` и стандартным отклонением (квадратный корень из дисперсии) `scale`.

In [ ]:
# Число наблюдений
n = 1000000

# Параметры
mu = 20                                  # математическое ожидание
sigma2 = 100                             # дисперсия

# Генерация
x = norm.rvs(size  = n,                  # число генерируемых наблюдений
             loc   = mu,                 # математическое ожидание
             scale = np.sqrt(sigma2))    # стандартное отклонение

# Посмотрим на первые несколько значений
print(x[0:10])

Функция `seaborn.histplot()` строит гистограмму.

In [ ]:
# Построим гистограмму
seaborn.histplot(x,                          # данные
                 stat = 'density',           # тип гистограммы
                 color = "palevioletred",    # цвет гистограммы
                 label = "histogram",        # наименование гистограммы
                 bins = 100)                 # число столбиков в Гистограмме

Превратим сгенерированные случайные величины $X$ в случайные величины $Y$ из экспоненциального распределения $D_{1}\sim \text{EXP}(0.1)$.

Функция `norm.cdf()` позволяет рассчитать значение функции распределения нормального распределения с математическим ожиданием `loc` и стандартным отклонением `scale`.

Функция `expon.ppf()` позволяет рассчитать значение квантильной функции экспоненциального распределения с математическим ожиданием `scale`. Напомним, что если экспоненциальная случайная величина имеет параметр $\lambda$, то ее математическое ожидание будет равняться $\frac{1}{\lambda}$.

В целом в библиотеке scipy у каждого распределения методы `cdf()` и `ppf() `отвечают за расчет функции распределения и квантильной функции соответственно, а метод` rvs()` - за генерацию случайных величин. Подробную документацию по этим методам можно найти в [документации](https://docs.scipy.org/doc/scipy/reference/stats.html).

In [ ]:
# Генерируем
lambda0 = 0.1
y = expon.ppf(norm.cdf(x,
                       loc = mu,
                       scale = np.sqrt(sigma2)),
              scale = 1 / lambda0)

# Посмотрим на несколько сгенерированных значений
print(y[0:10])

In [ ]:
# Построим гистограмму
seaborn.histplot(y,                          # данные
                 stat = 'density',           # тип гистограммы
                 color = "palevioletred",    # цвет гистограммы
                 label = "histogram",        # наименование гистограммы
                 bins = 100)                 # число столбиков в Гистограмме